# 🫁 PulmoScan AI — End-to-End Training & Evaluasi Klinis (Tesla T4 GPU)

Notebook ini menjalankan **seluruh tahapan riset dan pengembangan end-to-end** hanya dengan satu kali klik:
1. **Environment Setup:** Clone repository & instalasi dependensi.
2. **Dataset Acquisition:** Unduh otomatis dataset IQ-OTHNCCD (12.184 citra CT scan) via KaggleHub API.
3. **Stratified Split:** Pembagian data terstratifikasi (70% Train, 15% Validation, 15% Test).
4. **Deep Learning Baseline:** Training EfficientNet-B0 dengan **FP16 Mixed Precision** (`torch.cuda.amp`) di Tesla T4 GPU.
5. **Hybrid Machine Learning:** Ekstraksi 1.280-dim embedding & pelatihan Random Forest (balanced) dan XGBoost.
6. **Explainable AI (Grad-CAM):** Pembuatan peta atensi spasial area nodul kanker paru.
7. **Evaluasi Diagnostik:** Confusion Matrix, Multi-Class ROC-AUC Curve, dan 5-Fold Stratified Cross-Validation.
8. **Download Model:** Pengunduhan artefak model terbaik (`.pth` & `.joblib`).

### ⚡ Petunjuk Akselerator Hardware:
Pastikan akselerator GPU Tesla T4 telah aktif sebelum menjalankan:
Menu **Runtime** -> **Change runtime type** -> Pilih **T4 GPU** -> **Save**.

In [ ]:
# =============================================================================
# 1. Setup Repository, Direktori Kerja, & Dependensi
# =============================================================================
!rm -rf /content/lung-cancer-detection-ai
%cd /content
!git clone https://github.com/suzirz/lung-cancer-detection-ai.git
%cd /content/lung-cancer-detection-ai
!pip install -q -r requirements.txt

# Verifikasi GPU NVIDIA Tesla T4
!nvidia-smi

In [ ]:
# =============================================================================
# 2. Training Baseline CNN (EfficientNet-B0 FP16 di Tesla T4)
# =============================================================================
!PYTHONPATH=. python -m src.training.run_colab

In [ ]:
# =============================================================================
# 3. Ekstraksi Fitur Embedding & Training Model Hybrid (Random Forest & XGBoost)
# =============================================================================
!PYTHONPATH=. python -m src.training.train_hybrid

In [ ]:
# =============================================================================
# 4. Evaluasi Diagnostik Lanjutan (Multi-Class ROC-AUC & 5-Fold Cross-Validation)
# =============================================================================
!PYTHONPATH=. python -m src.evaluation.final_eval

In [ ]:
# =============================================================================
# 5. Tampilkan Seluruh Visualisasi & Grafik Laporan Evaluasi
# =============================================================================
import os
import json
from IPython.display import Image, display

print('======================================================================')
print('1. HASIL KOMPARASI PERFORMA MODEL (CNN vs Hybrid RF vs Hybrid XGBoost)')
print('======================================================================')
if os.path.exists('reports/model_comparison.png'):
    display(Image(filename='reports/model_comparison.png'))

if os.path.exists('reports/model_comparison.json'):
    with open('reports/model_comparison.json') as f:
        comp = json.load(f)
    print(json.dumps(comp, indent=2))

print('\n======================================================================')
print('2. CONFUSION MATRIX DIAGNOSTIK MEDIS')
print('======================================================================')
for cm_file in ['reports/baseline_confusion_matrix.png', 'reports/hybrid_random_forest_confusion_matrix.png']:
    if os.path.exists(cm_file):
        print(f'File: {cm_file}')
        display(Image(filename=cm_file))

print('\n======================================================================')
print('3. MULTI-CLASS ROC-AUC CURVE')
print('======================================================================')
if os.path.exists('reports/roc_auc_curve.png'):
    display(Image(filename='reports/roc_auc_curve.png'))

In [ ]:
# =============================================================================
# 6. Unduh Bobot Model Terbaik ke Komputer Anda
# =============================================================================
from google.colab import files

for path in [
    'models/baseline_efficientnet_b0_best.pth',
    'models/hybrid_random_forest.joblib',
    'models/hybrid_xgboost.joblib'
]:
    if os.path.exists(path):
        print(f'Mengunduh {path}...')
        files.download(path)